In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

In [ ]:
import os

# Mencari file data.yaml di seluruh Google Drive kamu
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'data.yaml' in files:
        print("✅ LOKASI DATA.YAML DITEMUKAN DI:")
        print(os.path.join(root, 'data.yaml'))
        print("-" * 50)

In [ ]:
from ultralytics import YOLO

yaml_path = '/content/drive/MyDrive/Rock Paper Scissors SXSW.v14i.yolov8/data.yaml'

# Load model dasar (Pre-trained YOLOv8 Nano)
model = YOLO('yolov8n.pt')

# Mulai Training
results = model.train(
    data=yaml_path,     # Menggunakan file data.yaml yang ada di Drive
    epochs=25,          # Jumlah putaran belajar (bisa dinaikkan ke 30-50 jika waktu cukup)
    imgsz=640,          # Resolusi gambar standar YOLO
    batch=16,           # Jumlah gambar yang diproses per batch
    name='model_rock_paper_scissors' # Nama folder penyimpanan hasil
)

In [ ]:
import shutil

# Menyalin model terbaik dari penyimpanan sementara Colab langsung ke Google Drive
lokasi_asal = '/content/runs/detect/model_rock_paper_scissors/weights/best.pt'
lokasi_tujuan = '/content/drive/MyDrive/best_model_rock_paper_scissors.pt'

shutil.copy(lokasi_asal, lokasi_tujuan)
print(f"✅ Model berhasil diamankan ke Google Drive di: {lokasi_tujuan}")

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
from ultralytics import YOLO

# 1. Load model terbaik yang sudah kamu latih
model_ku = YOLO('/content/drive/MyDrive/best_model_rock_paper_scissors.pt')

# 2. Masukkan jalur salah satu gambar dari folder test di Drive kamu
gambar_test = '/content/drive/MyDrive/JALUR_KE_FOLDER_DATASET/test/images/nama_gambar_contoh.jpg'

# 3. Lakukan deteksi
hasil = model_ku(gambar_test)

# 4. Tampilkan gambarnya beserta kotak deteksi (bounding box)
cv2_imshow(hasil[0].plot())

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
from google.colab.patches import cv2_imshow
from ultralytics import YOLO

# 1. Fungsi untuk mengaktifkan Webcam di browser Colab
def ambil_foto(filename='foto_webcam.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Ambil Foto';
      capture.style.padding = '10px 20px';
      capture.style.fontSize = '16px';
      capture.style.backgroundColor = '#4CAF50';
      capture.style.color = 'white';
      capture.style.border = 'none';
      capture.style.borderRadius = '5px';
      capture.style.cursor = 'pointer';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.marginTop = '10px';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Sesuaikan ukuran frame
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Tunggu sampai tombol diklik
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# ==========================================
# --- MULAI PROSES DETEKSI VIA WEBCAM ---
# ==========================================

try:
  print("🔄 Mengaktifkan kamera... (Jangan lupa klik 'Allow/Izinkan' di browsermu)")

  # 2. Ambil foto dari kamera laptop
  filename = ambil_foto()
  print("✅ Foto berhasil diambil! Model sedang menganalisis jari tanganmu...")

  # 3. Load model terbaik yang sudah kamu simpan di Drive
  model_ku = YOLO('/content/drive/MyDrive/best_model_rock_paper_scissors.pt')

  # 4. Lakukan deteksi pada foto yang baru saja dijepret
  hasil = model_ku(filename)

  # 5. Tampilkan hasil foto beserta kotak deteksinya (Rock / Paper / Scissors)
  print("\n--- HASIL DETEKSI ---")
  cv2_imshow(hasil[0].plot())

except Exception as err:
  print(f"❌ Terjadi kesalahan atau kamera dibatalkan: {str(err)}")

In [ ]:
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from base64 import b64decode
import cv2
import numpy as np
from ultralytics import YOLO

# 1. Load model terbaikmu dari Google Drive
model_ku = YOLO('/content/drive/MyDrive/best_model_rock_paper_scissors.pt')

# 2. Fungsi untuk mengambil video stream dari webcam browser
def ambil_frame_live():
  js = Javascript('''
    async function getFrame() {
      if (!window.streamVideo) {
        const video = document.createElement('video');
        video.style.display = 'none';
        window.streamVideo = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(video);
        video.srcObject = window.streamVideo;
        await video.play();
        window.videoEl = video;
      }
      const canvas = document.createElement('canvas');
      canvas.width = window.videoEl.videoWidth;
      canvas.height = window.videoEl.videoHeight;
      canvas.getContext('2d').drawImage(window.videoEl, 0, 0);
      return canvas.toDataURL('image/jpeg', 0.7);
    }
    ''')
  display(js)
  data = eval_js('getFrame()')
  binary = b64decode(data.split(',')[1])
  image_np = np.frombuffer(binary, dtype=np.uint8)
  return cv2.imdecode(image_np, flags=1)

print("🔄 Mengaktifkan kamera real-time... (Klik 'Allow/Izinkan' di browser)")
print("🛑 TEKAN TOMBOL STOP ⏹️ DI COLAB UNTUK MENGHENTIKAN VIDEO!")

# 3. Loop Video Real-Time
try:
  while True:
    # Ambil frame dari kamera
    frame = ambil_frame_live()

    # Lakukan deteksi (verbose=False agar log tidak memenuhi layar)
    hasil = model_ku(frame, verbose=False)

    # Bersihkan layar sebelumnya dan tampilkan frame terbaru
    clear_output(wait=True)
    print("🔴 LIVE STREAMING - Tekan tombol Stop ⏹️ di Colab untuk berhenti")
    cv2_imshow(hasil[0].plot())

except KeyboardInterrupt:
  print("\n🛑 Deteksi real-time berhasil dihentikan.")